# GENESIS — Gemma 4 Experiments

Quick notebook to test and benchmark Gemma 4 vs Gemma 3 inference.

In [ ]:
import sys
sys.path.insert(0, '..')
import os
from dotenv import load_dotenv
load_dotenv('../.env')
HF_TOKEN = os.getenv('HF_TOKEN', '')
print('HF_TOKEN set:', bool(HF_TOKEN))

In [ ]:
from core.gemma_engine import GemmaEngine, GEMMA_MODELS
print('Available models:', list(GEMMA_MODELS.keys()))

In [ ]:
# Gemma 4 — primary
engine_g4 = GemmaEngine(token=HF_TOKEN, model='default')
print(engine_g4)

In [ ]:
# Gemma 3 — fallback
engine_g3 = GemmaEngine(token=HF_TOKEN, model='g3')
print(engine_g3)

In [ ]:
import time

prompt = 'Explain transformer attention in 3 sentences.'

t0 = time.time()
r4 = engine_g4.think(prompt, max_tokens=200)
t4 = time.time() - t0

t0 = time.time()
r3 = engine_g3.think(prompt, max_tokens=200)
t3 = time.time() - t0

print(f'Gemma 4 ({t4:.1f}s):\n{r4}\n')
print(f'Gemma 3 ({t3:.1f}s):\n{r3}')

In [ ]:
# JSON structured output test
schema = '{"topic": str, "summary": str, "key_points": [str]}'
result = engine_g4.think_json(
    'Summarise transformer attention',
    schema_hint=schema,
)
import json; print(json.loads(result))

In [ ]:
# Async batch test
import asyncio

prompts = [
    'What is backpropagation?',
    'What is gradient descent?',
    'What is a learning rate?',
]

async def run_batch():
    return await engine_g4.think_batch_async(prompts, max_tokens=80)

results = asyncio.run(run_batch())
for q, a in zip(prompts, results):
    print(f'Q: {q}\nA: {a[:100]}...\n')